# Lezione 15 — Tokenizzazione: il modello legge il testo

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezioni 5, 10, 12 - encoding e scaling (Lezione 5), la prima rete Keras (Lezione 10), il modello finale valutato onestamente (Lezione 12). Qui comincia la **Fase 3** (testo ed embedding) — il cuore del Memory
AI Lab.

**Cosa saprai fare alla fine:** trasformare testo in input numerici con
tokenizzazione e vocabolario, conoscere i limiti della rappresentazione "a
conteggi", e capire perché finora il classificatore era cieco.

## Parte 1 — Teoria: dal testo ai numeri

I modelli calcolano, non leggono (il principio della Lezione 5): il testo deve diventare numeri. La catena
standard ha tre anelli:

1. **Tokenizzazione** — spezzare il testo in unità (*token*). La versione
   semplice: minuscole, via la punteggiatura, split sugli spazi.
   `"Marco visited Glasgow."` → `["marco", "visited", "glasgow"]`.
   (I modelli moderni usano *sub-word*: pezzi di parola. Lo vedrai con i
   Transformer in Fase 5 — il principio è identico.)

2. **Vocabolario** — la mappa token → intero, costruita **sul train**: il vocabolario è una statistica appresa dai dati, esattamente come la mediana della Lezione 4, e va calcolato solo sul train per non far filtrare informazione dal test. Le parole mai viste diventano il token speciale **OOV** (*out of vocabulary*): in produzione arrivano sempre parole nuove, e devono avere un posto previsto - lo stesso ruolo del `reindex` della Lezione 5 per le colonne one-hot mai viste nel train, qui applicato al testo.

3. **Rappresentazione** — la più semplice è il **bag of words**: per ogni
   frase, quali parole del vocabolario contiene (multi-hot: 0/1 per ogni
   parola). Butta via l'**ordine** ("Marco chiama Lucia" = "Lucia chiama
   Marco") e non sa che "treno" e "ferrovia" si somigliano — limiti seri,
   che gli embedding (prossima lezione) risolveranno. Ma per capire *di
   che tipo* è una frase, le parole presenti bastano spesso a fare molto.

In Keras tutto questo è un layer: `TextVectorization`.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import keras

keras.utils.set_random_seed(42)
print('Keras', keras.__version__)

In [ ]:
from keras.layers import TextVectorization

frasi_demo = np.array(['Marco visited Glasgow with his son.',
                       'The user prefers short updates.'])

vettorizzatore_demo = TextVectorization(max_tokens=20, output_mode='int')
vettorizzatore_demo.adapt(frasi_demo)

print('vocabolario:', vettorizzatore_demo.get_vocabulary())
print('\nprima frase come id di token:')
print(vettorizzatore_demo(frasi_demo)[0].numpy())

Leggi l'output: il vocabolario ha in testa `''` (padding) e `'[UNK]'`
(l'OOV), poi i token per frequenza; ogni frase è diventata una sequenza di
interi. Con `output_mode='multi_hot'` otterremmo invece il bag of words:
un vettore 0/1 lungo quanto il vocabolario.

## Parte 2 — Esercizio guidato

Il tuo compito, con un nuovo `TextVectorization(max_tokens=20,
output_mode='int')` adattato sulle stesse due frasi:

1. trasforma la frase nuova `'Marco prefers trains.'`;
2. identifica nell'output quali token sono finiti su `[UNK]` (id 1) e
   spiega perché;
3. verifica che `'marco'` e `'Marco'` producano lo stesso id (cosa fa la
   standardizzazione di default?).

**Prova tu:**

In [ ]:
# Scrivi qui: adapt su frasi_demo, transform di 'Marco prefers trains.',
# poi confronta con il vocabolario stampato sopra.

pass

### Soluzione spiegata

- `marco` e `prefers` sono nel vocabolario (comparivano nel train);
  `trains` no → id 1, `[UNK]`: il posto previsto per le parole nuove;
- la standardizzazione di default abbassa le maiuscole e toglie la
  punteggiatura: `'Marco'` e `'marco'` coincidono — lo stesso principio della normalizzazione dei near-duplicates della Lezione 2 (`strip().casefold()` sulle stazioni), qui integrato nel layer;
- nota il principio di fondo: **nessuna parola nuova rompe la pipeline**,
  tutto ha una rappresentazione prevista.

In [ ]:
vet = TextVectorization(max_tokens=20, output_mode='int')
vet.adapt(frasi_demo)
vocabolario = vet.get_vocabulary()

ids = vet(np.array(['Marco prefers trains.']))[0].numpy()
print('id prodotti:', ids)
print('token      :', [vocabolario[i] for i in ids if i > 0])
assert vet(np.array(['marco']))[0].numpy()[0] == vet(np.array(['Marco']))[0].numpy()[0]

## Parte 3 — Il progetto: Memory AI Lab, passo 15 — il modello legge

Il momento della verità. Finora il classificatore vedeva 7 numeri poveri (lunghezze e flag) e si fermava intorno al 60-65% — e le Lezioni 10 e 12 lo avevano diagnosticato guardando il divario tra la rete e la baseline lineare: *il collo di bottiglia sono le feature*, non il modello. Oggi il
modello legge le **parole** delle memorie: vocabolario costruito sul train
(niente leakage), bag of words multi-hot, stessa rete, stesso protocollo.

Se la diagnosi era giusta, l'accuratezza deve saltare.

In [ ]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
memorie = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
classi = ['episodic', 'preference', 'semantic', 'unknown']
mappa = {c: i for i, c in enumerate(classi)}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie.items()}
target = {n: f['type'].map(mappa).to_numpy() for n, f in memorie.items()}

vettorizzatore = TextVectorization(max_tokens=300, output_mode='multi_hot')
vettorizzatore.adapt(testi['train'])                 # vocabolario dal SOLO train
X_testo = {n: vettorizzatore(t).numpy() for n, t in testi.items()}
print('bag of words:', X_testo['train'].shape, '(memorie x parole del vocabolario)')

In [ ]:
keras.utils.set_random_seed(42)
lettore = keras.Sequential([
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(4, activation='softmax'),
])
lettore.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                metrics=['accuracy'])
lettore.fit(X_testo['train'], target['train'], epochs=300,
            validation_data=(X_testo['val'], target['val']),
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                     restore_best_weights=True)],
            verbose=0)

acc_testo = lettore.evaluate(X_testo['val'], target['val'], verbose=0)[1]
print('feature povere (Lezioni 10-12) : ~60-65% su validation')
print(f'leggendo il testo              : {acc_testo:.0%} su validation')

Eccolo, il salto — e la conferma che il metodo del corso funziona: la
diagnosi onesta delle Lezioni 10-12 ("non è il modello, sono le feature")
indicava esattamente qui. Le parole di una memoria dicono *che tipo* di
memoria è: "prefers/likes" → preference, "visited/booked" → episodic.

Ma il bag of words ha i suoi limiti strutturali, già annunciati: non sa che
"train" e "railway" si somigliano, ignora l'ordine, e con vocabolari veri
(centinaia di migliaia di parole) esplode in dimensione. La soluzione ha un
nome: **embedding** — la prossima lezione, e il cuore di tutto quello che
segue nel corso.

## Cosa hai imparato — e la Fase 3 è aperta

- La catena testo → numeri: **tokenizzazione** → **vocabolario** (costruito
  sul solo train) → rappresentazione.
- L'**OOV** dà un posto previsto alle parole mai viste: niente rompe la
  pipeline in produzione.
- Il **bag of words** cattura *quali* parole ci sono (spesso basta per il
  tipo), ma ignora ordine e somiglianza tra parole.
- La diagnosi "collo di bottiglia = feature" era giusta: il testo ha fatto
  saltare l'accuratezza dove il modello da solo non poteva.

## Quiz

1. Perché il vocabolario si costruisce (adapt) sul solo train?
2. Una memoria in produzione contiene una parola mai vista. Cosa succede?
3. Con il bag of words, "Lucia booked a train to Milano" e "Milano booked
   a train to Lucia" hanno la stessa rappresentazione. Perché, e quale
   informazione serve per distinguerle?

<details>
<summary><b>Apri le risposte</b></summary>

1. È una statistica appresa dai dati, come la mediana della Lezione 4:
   costruirlo su tutto includerebbe nel train informazione su validation e
   test (quali parole esistono, con che frequenze) — leakage da
   preprocessing.
2. Diventa il token OOV `[UNK]`: rappresentazione prevista, pipeline
   intatta, il modello la tratta come "parola sconosciuta".
3. Il multi-hot registra solo *quali* parole compaiono, non dove: le due
   frasi hanno le stesse parole. Per distinguerle serve l'ordine — cioè
   rappresentazioni sequenziali, che arrivano con embedding + modelli
   sequenziali (e poi l'attention, in Fase 5).
</details>

## Fonti

- Keras, `TextVectorization`:
  https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/
- TensorFlow, *Basic text classification*:
  https://www.tensorflow.org/tutorials/keras/text_classification

## Prossima lezione

Ogni parola diventa un **vettore denso imparato dal modello**, dove le
parole simili stanno vicine: l'embedding. È l'idea che rende possibile la
ricerca semantica del Memory AI Lab — e tutto il resto del corso.